# LatentR3: SFT con token latente + LR-GRPO

Xu et al. (2025), *Reinforced Latent Reasoning for LLM-based Recommendation*. Sobre Qwen2.5-1.5B se agrega una capa de atención, **LatentRATT**, que produce un token latente de razonamiento a partir de los estados ocultos del prompt; el modelo usa ese token para escribir el título del siguiente ítem. Se entrena en dos etapas: un **SFT** de todo el modelo y un **RL (LR-GRPO)** que solo ajusta LatentRATT, con muestras del token latente perturbadas con ruido, recompensa −PPL del objetivo y ventaja calculada sobre un batch de grupos.

Se usa el código de los autores ([xuwenxinedu/R3](https://github.com/xuwenxinedu/R3), copia en `repo_latentr3/`, commit `c558b44`), verificado por sha256. Dos archivos se usan desde una copia parcheada en `autores/` (diff en `autores/cambios.diff`):
- **P1:** se quita un import de un módulo que no existe (`noisy_layer`).
- **P2:** el primer pase de LatentRATT no calcula logits que se descartaban (solo memoria).
- **P3:** "2 GPUs simuladas": el RL se comporta como con las 2 GPUs de los autores en 1 GPU.
- **P5:** los dos productos de atención de LatentRATT se calculan elemento a elemento cuando hay gradiente, para esquivar un defecto de hipBLASLt en ROCm 7.14 (mismo resultado).

El SFT usa **AdamW con los estados en la RAM** (F3), el mismo algoritmo, para caber en 16 GB. Las decisiones, los costos y los problemas están en [`README.md`](README.md).

Cada dataset tiene su propio modelo, entrenado desde Qwen2.5-1.5B base, y se entrena y evalúa por separado: solo se corren los de `DATASETS`. **Este notebook escribe archivos** en `checkpoints/latentr3/<categoría>/` (ignorado por git) y puede retomarse si se corta.

En local se ejecuta desde `notebooks/latentr3/` y lee los datos desde `../../repo_latentr3/data`.

## Entorno y configuración

In [1]:
import os
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if not IN_COLAB:
    os.environ['HF_HUB_OFFLINE'] = '1'   # Qwen2.5-1.5B ya está en la caché local: no se descarga nada
    # Con una GPU integrada, PyTorch ve dos GPUs y el Trainer reparte el modelo entre ambas (DataParallel), lo que
    # agota la memoria. Se deja visible solo la dedicada (índice 0). Tiene que ir antes de importar torch; no se
    # combina con HIP_VISIBLE_DEVICES (README de la raíz)
    os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

import gc
import json
import shutil
import pandas as pd
import torch
import transformers
import trl

E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48


In [2]:
DATASETS = ['toys_data']   # se entrenan y evalúan solo estos (nombres de `names`)
names = {'CDs_and_Vinyl': 'cd_data',
         'Musical_Instruments': 'music_data',
         'Toys_and_Games': 'toys_data',
         'Video_Games': 'games_data'}
categorias = {nombre: categoria for categoria, nombre in names.items()}

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
import sys
base = 'data_amazon_2018' if IN_COLAB else os.path.join('..', '..', 'repo_latentr3', 'data')
latentr3 = os.path.abspath('repo_latentr3' if IN_COLAB else os.path.join('..', '..', 'repo_latentr3'))
codigo_d3 = os.path.abspath('DecodingMatters' if IN_COLAB else os.path.join('..', '..', 'DecodingMatters'))
SALIDA = ('/content/drive/MyDrive/IIC3633/checkpoints/latentr3' if IN_COLAB
          else os.path.abspath(os.path.join('..', '..', 'checkpoints', 'latentr3')))
SALIDA_BIGREC = os.path.join(os.path.dirname(SALIDA), 'bigrec')   # para comparar con nuestro BIGRec y D3
calc_py = os.path.join(latentr3, 'src', 'utils', 'calc.py')
# latentr3.py importa baseline_comun.py (notebooks/) y bigrec_baseline.py (notebooks/bigrec/)
for carpeta in (['.'] if IN_COLAB else ['..', os.path.join('..', 'bigrec')]):
    carpeta = os.path.abspath(carpeta)
    if carpeta not in sys.path:
        sys.path.insert(0, carpeta)

In [3]:
for archivo in [calc_py, 'latentr3.py', os.path.join('autores', 'model.py'), os.path.join('autores', 'grpo_trainer.py')]:
    if not os.path.isfile(archivo):
        raise FileNotFoundError(f'No se encontró {os.path.abspath(archivo)}.')

import latentr3 as lr3
import bigrec_baseline as bb
lr3.cargar_autores(latentr3)   # verifica los sha256 (originales y copias parcheadas) antes de importar
CONFIG_SFT, CONFIG_RL = lr3.CONFIG_SFT, lr3.CONFIG_RL
print('Configuración de memoria. SFT:', CONFIG_SFT, '| RL:', CONFIG_RL)

print('torch', torch.__version__, '| transformers', transformers.__version__, '| trl', trl.__version__)
if transformers.__version__ != '4.51.3' or trl.__version__ != '0.16.1':
    print('AVISO: los autores usan transformers 4.51.3 y trl 0.16.1; el trainer parcheado depende de esa versión de TRL.')
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), '| bf16:', torch.cuda.is_bf16_supported())
else:
    print('Sin GPU: solo se pueden correr las verificaciones sin GPU.')

Configuración de memoria. SFT: {'micro_batch': 2, 'checkpointing': True, 'optim_cpu': True} | RL: {'checkpointing': False}
torch 2.14.1+rocm7.14 | transformers 4.51.3 | trl 0.16.1
AMD Radeon RX 9070 XT | bf16: True


## Datos

Los mismos CSV que los baselines, sin re-split. Se comparan los conteos con la Tabla 3 del paper (en CDs, `test` tiene 6157 filas y el paper dice 6158).

In [5]:
TABLA3 = {'toys_data': (53898, 6737, 6738, 6299), 'cd_data': (49251, 6156, 6158, 5841),
          'games_data': (75175, 9397, 9397, 5308), 'music_data': (66500, 8312, 8313, 5030)}
conteos = {}
for nombre in DATASETS:
    categoria = categorias[nombre]
    fila = {split: len(pd.read_csv(lr3.buscar_archivo(base, split, categoria))) for split in ['train', 'valid', 'test']}
    with open(lr3.buscar_archivo(base, 'info', categoria), encoding='utf-8') as f:
        fila['ítems'] = sum(1 for _ in f)
    fila['Tabla 3 (train, valid, test, ítems)'] = TABLA3.get(nombre)
    conteos[nombre] = fila
pd.DataFrame(conteos).T

,train,valid,test,ítems,"Tabla 3 (train, valid, test, ítems)"
toys_data,53898,6737,6738,6299,"(53898, 6737, 6738, 6299)"


## Verificaciones sin GPU (V1–V9)

Corren en la CPU, con modelos diminutos (2 capas, dimensión 64) que sirven para comprobar el código, no la calidad. Comprueban los sha256 y `cambios.diff` (V1), el token latente (V2), el prompt y el trie (V3, V4), LatentRATT con P2 y P5 (V5), el congelamiento del RL (V6), P3 y el sampler de las 2 GPUs simuladas (V7, V8) y que `cutoff_len` 512 no trunca (V9).

El detalle de cada una está en [`README.md`](README.md), sección "Adaptaciones respecto de la fuente".

In [6]:
print('V1', lr3.verificar_codigo(latentr3))
tokenizer = lr3.tokenizer_con_latente()
print('V2', lr3.verificar_token(tokenizer))
hay_d3 = os.path.isfile(os.path.join(codigo_d3, 'code', 'dataset.py'))
if hay_d3:
    bb.cargar_d3(codigo_d3)
for nombre in DATASETS:
    ejemplo = lr3.verificar_prompt_latente(base, categorias[nombre], tokenizer)
    if hay_d3:
        bb.verificar_prompt(base, categorias[nombre], latentr3)
    print(f'V3 {nombre}: prompt del SFT = prompt del RL{" = BIGRec + <|Thought|>" if hay_d3 else ""} en 200 filas por split')
print('\nEjemplo de train (con el objetivo y EOS):\n')
print(ejemplo)

V1 {'originales': 8, 'copias verificadas': ['model.py', 'grpo_trainer.py'], 'líneas con PARCHE en el diff': 9}


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


V2 {'id de <|Thought|>': 151665, 'len(tokenizer)': 151666, 'filas de embeddings tras el resize': 151666}


100%|██████████| 1/1 [00:00<00:00, 1440.35it/s]


V3 toys_data: prompt del SFT = prompt del RL = BIGRec + <|Thought|> en 200 filas por split

Ejemplo de train (con el objetivo y EOS):

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request. 

### Instruction:
Given a list of toys and games the user recetenly enjoy, please write a new toys and games that the user may bought
### User Input: 
The user has palyed the following toys and gamess before: "Melissa & Doug Wooden Jigsaw Puzzles Set: Vehicles and Construction"

### Response: 
<|Thought|>"Alex Discover My Giant Busy Box"
<|endoftext|>


In [7]:
verificacion, largos = {}, {}
for nombre in DATASETS:
    categoria = categorias[nombre]
    test = lr3.dataset_sft(tokenizer, lr3.buscar_archivo(base, 'test', categoria), categoria, test=True)
    verificacion[nombre] = lr3.verificar_trie_latente(lr3.buscar_archivo(base, 'info', categoria), tokenizer,
                                                      [test[i] for i in range(len(test))])
    train = lr3.dataset_sft(tokenizer, lr3.buscar_archivo(base, 'train', categoria), categoria)
    largos[(nombre, 'train: prompt + objetivo')] = lr3.largos_tokens(train)
    largos[(nombre, 'test: prompt')] = lr3.largos_tokens(test)
    assert largos[(nombre, 'train: prompt + objetivo')]['máximo'] < lr3.CUTOFF_LEN
print('V4 y V9')
display(pd.DataFrame(verificacion).T)
pd.DataFrame(largos).T.round(1)

100%|██████████| 53898/53898 [00:16<00:00, 3297.47it/s]

V4 y V9


,ítems,títulos únicos,caminos del trie,tokens del título más largo,títulos con más de 128 tokens
toys_data,6299,6271,6271,69,0


filas  media    p99  máximo
toys_data train: prompt + objetivo  53898.0  155.9  281.0   366.0
          test: prompt               6738.0  160.2  269.6   340.0

In [8]:
print('V5', lr3.verificar_latentratt(latentr3))
print('V6', lr3.verificar_congelamiento())
print('V7', lr3.verificar_p3(base, categorias[DATASETS[0]], lr3.tokenizer_con_latente(), os.path.join(SALIDA, '_v7')))
print('V8', lr3.verificar_sampler())

Sliding Window Attention is enabled but not implemented for `eager`; unexpected results may be encountered.


V5 {'parámetros de LatentRATT': 12480, 'esperado 3(d²+d)': 12480, 'P5: dif. máx. salida (con vs. sin gradiente)': 1.1920928955078125e-07, 'P5: dif. máx. salida vs. original': 1.1920928955078125e-07, 'P5: dif. máx. gradiente vs. original': 4.76837158203125e-07, 'padding: dif. máx. de r': 1.4901161193847656e-07}
V6 {'parámetros entrenables': 12480, 'tensores cambiados tras un paso': ['attention.query.weight', 'attention.query.bias', 'attention.key.weight', 'attention.key.bias', 'attention.value.weight', 'attention.value.bias']}


100%|██████████| 2/2 [00:00<00:00, 1410.32it/s]


V7 {'grupos por micro-paso': 2, 'ventaja: dif. máx. vs. Eq. 7': 0.0, 'gradiente: dif. máx. por grupo vs. un grafo': 0.0, 'swap con 1 grupo falla': True}
V8 {'filas muestreadas': 7936, 'micro-pasos': 496, 'pasos de optimización': 31, 'primer micro-paso': [542, 542, 542, 542, 542, 542, 542, 542, 618, 618, 618, 618, 618, 618, 618, 618]}


## Prueba rápida

En Toys, en `checkpoints/latentr3/_prueba/`, que se borra al final. La calidad del modelo no importa: sirve para comprobar el pipeline completo y medir tiempos con la configuración elegida.
1. **SFT** con 1024 filas de `train` (8 pasos de 128 filas por época, hasta 3 épocas) y 256 de `valid`.
2. **Evaluación del SFT** en 100 filas de `test` con lotes de 24: 0 títulos fuera del catálogo y métricas iguales a las de `calc.py`.
3. **RL** desde ese modelo con 64 prompts (2 pasos de 256 filas). Se comprueba tensor por tensor que solo cambió LatentRATT.
4. **Evaluación del RL** en 100 filas con lotes de 12, con `length_penalty` 1,0 (LatentR3) y 0,0 (D3 + LatentR3).
5. **Estimación** de horas por dataset con lo medido.

La memoria en el peor caso (las 256 filas más largas de Instruments, 448 tokens) se midió antes: 9,8 GB en el SFT con F3 (medido fuera del notebook, sin output guardado).

In [9]:
PRUEBA = os.path.join(SALIDA, '_prueba')
categoria_prueba = 'Toys_and_Games'
rp = lr3.rutas(PRUEBA)
registro_sft = lr3.entrenar_sft(categoria_prueba, base, rp['sft'], sample_train=1024, sample_valid=256, **CONFIG_SFT)
pd.Series({k: registro_sft[k] for k in ['s_por_paso_mediana', 's_valid_por_fila', 'memoria_max_gb',
                                         'memoria_reservada_max_gb', 'micro_batch', 'acumulacion', 'optimizador']})

100%|██████████| 256/256 [00:00<00:00, 2970.00it/s]
Some weights of LatentModel were not initialized from the model checkpoint at Qwen/Qwen2.5-1.5B and are newly initialized: ['attention.key.bias', 'attention.key.weight', 'attention.query.bias', 'attention.query.weight', 'attention.value.bias', 'attention.value.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.039600,3.082762
2,2.318100,2.980563
3,1.419900,3.138953


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


s_por_paso_mediana                19.366606
s_valid_por_fila                    0.02212
memoria_max_gb                     9.733309
memoria_reservada_max_gb          14.146484
micro_batch                               2
acumulacion                              64
optimizador                 AdamWEnCPU (F3)
dtype: object

In [10]:
info_prueba = lr3.buscar_archivo(base, 'info', categoria_prueba)
eval_prueba = {}
def evaluar_prueba(nombre, dir_modelo, dir_pred, lote, lp):
    ruta = lr3.evaluar_categoria(categoria_prueba, base, dir_modelo, dir_pred, lote, lp, sample=100)
    with open(os.path.join(dir_pred, 'registro.json')) as f:
        registro = json.load(f)
    lr3.verificar_con_calc(ruta, info_prueba, calc_py)   # falla si las métricas no coinciden con calc.py
    eval_prueba[nombre] = {'s por fila': registro['segundos'] / registro['filas'],
                           'memoria máx. GB': registro.get('memoria_max_gb'), **lr3.metricas(ruta, info_prueba)}
evaluar_prueba('SFT (lotes de 24)', rp['modelo_sft'], rp['pred_sft'], lr3.BATCH_GEN_SFT, lr3.LENGTH_PENALTY['latentr3'])
pd.DataFrame(eval_prueba).T

100%|██████████| 100/100 [00:00<00:00, 2492.57it/s]
`generation_config` default values have been modified to match model-specific defaults: {'bos_token_id': 151643}. If this is not desired, please set these values explicitly.
[W1124 23:26:14.732455731 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 1128267776 bytes (free: 578813952, total: 17095983104).
[W1124 23:26:23.165687622 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 1145044992 bytes (free: 486539264, total: 17095983104).


Toys_and_Games: parte 1/1, 0.625 s por fila


,s por fila,memoria máx. GB,HR@1,HR@3,HR@5,HR@10,HR@20,NDCG@1,NDCG@3,NDCG@5,NDCG@10,NDCG@20,fuera del catálogo,repetidos,filas
SFT (lotes de 24),0.624834,13.460786,0.01,0.04,0.04,0.04,0.04,0.01,0.027619,0.027619,0.027619,0.027619,0.0,0.0,100.0


In [11]:
registro_rl = lr3.entrenar_rl(categoria_prueba, base, rp['rl'], rp['modelo_sft'], sample=64, **CONFIG_RL)
print('Solo cambió LatentRATT:', lr3.verificar_rl_congelado(rp['modelo_sft'], rp['modelo_rl']))
pd.Series({k: registro_rl[k] for k in ['pasos_totales', 's_por_paso_mediana', 'memoria_max_gb', 'parametros_entrenables',
                                        'recompensa_por_paso', 'micro_batch', 'acumulacion']})

100%|██████████| 64/64 [00:00<00:00, 2388.62it/s]


Step,Training Loss
1,-0.001100
2,-0.002200


Solo cambió LatentRATT: {'tensores iguales': 338, 'tensores cambiados': ['attention.key.bias', 'attention.key.weight', 'attention.query.bias', 'attention.query.weight', 'attention.value.bias', 'attention.value.weight']}


pasos_totales                                                    2
s_por_paso_mediana                                        9.796339
memoria_max_gb                                            8.519104
parametros_entrenables                                     7082496
recompensa_por_paso       [-49.92501184344292, -62.91029170155525]
micro_batch                                                     16
acumulacion                                                     16
dtype: object

In [12]:
evaluar_prueba('RL (lotes de 12)', rp['modelo_rl'], rp['pred_rl'], lr3.BATCH_GEN_RL, lr3.LENGTH_PENALTY['latentr3'])
evaluar_prueba('RL + D3 (lotes de 12)', rp['modelo_rl'], rp['pred_d3'], lr3.BATCH_GEN_RL, lr3.LENGTH_PENALTY['d3'])
assert all(r['fuera del catálogo'] == 0 for r in eval_prueba.values())
pd.DataFrame(eval_prueba).T

100%|██████████| 100/100 [00:00<00:00, 2261.14it/s]


Toys_and_Games: parte 1/1, 0.391 s por fila


100%|██████████| 100/100 [00:00<00:00, 2409.91it/s]


Toys_and_Games: parte 1/1, 0.372 s por fila


,s por fila,memoria máx. GB,HR@1,HR@3,HR@5,HR@10,HR@20,NDCG@1,NDCG@3,NDCG@5,NDCG@10,NDCG@20,fuera del catálogo,repetidos,filas
SFT (lotes de 24),0.624834,13.460786,0.01,0.04,0.04,0.04,0.04,0.01,0.027619,0.027619,0.027619,0.027619,0.0,0.0,100.0
RL (lotes de 12),0.391098,11.287307,0.01,0.04,0.04,0.04,0.04,0.01,0.027619,0.027619,0.027619,0.027619,0.0,0.0,100.0
RL + D3 (lotes de 12),0.371735,8.245703,0.01,0.03,0.04,0.04,0.04,0.01,0.021309,0.025178,0.025178,0.025178,0.0,0.0,100.0


In [13]:
# Estimación por dataset con lo medido (la generación no se escala por el largo del prompt)
muestra = lr3.dataset_sft(tokenizer, lr3.buscar_archivo(base, 'train', categoria_prueba), categoria_prueba, sample=1024)
medicion = {'tokens_por_fila': lr3.largos_tokens(muestra)['media'],
            's_por_paso_sft': registro_sft['s_por_paso_mediana'], 's_valid_por_fila': registro_sft['s_valid_por_fila'],
            's_por_paso_rl': registro_rl['s_por_paso_mediana'],
            's_test_por_fila_sft': eval_prueba['SFT (lotes de 24)']['s por fila'],
            's_test_por_fila_rl': eval_prueba['RL (lotes de 12)']['s por fila']}
print('Medido:', medicion)
estimaciones = {names[c]: lr3.estimar(c, medicion) for c in lr3.FILAS}
BORRAR_PRUEBA = True
if BORRAR_PRUEBA:
    shutil.rmtree(PRUEBA, ignore_errors=True)
tabla = pd.DataFrame(estimaciones).T
tabla.loc['total'] = tabla.sum()
tabla.round(2)

100%|██████████| 1024/1024 [00:00<00:00, 2372.00it/s]


Medido: {'tokens_por_fila': np.float64(155.62890625), 's_por_paso_sft': 19.36660647392273, 's_valid_por_fila': 0.0221203125, 's_por_paso_rl': 9.7963387966156, 's_test_por_fila_sft': 0.6248339867591858, 's_test_por_fila_rl': 0.3910978865623474}


,h SFT (3 épocas),h RL (1 época),h evaluación (SFT + RL + D3),h total
cd_data,5.13,3.40,2.41,10.93
toys_data,6.93,4.59,2.63,14.16
games_data,9.33,6.18,3.67,19.19
music_data,10.59,7.01,3.25,20.85
total,31.98,21.18,11.96,65.13


## Etapa 1: SFT

Un modelo por dataset, desde Qwen2.5-1.5B base con `<|Thought|>` y LatentRATT recién inicializada. Ajusta todo el modelo con next-token prediction sobre el título objetivo (Eq. 4): AdamW, lr 3e-4, 20 pasos de warmup y bajada lineal, batch de 128 filas, hasta 3 épocas con early stopping de paciencia 1 sobre la loss de `valid`, y se queda con la mejor época (`latent_attention_train.py`; el autor lo confirma en el issue #2).

Memoria en 16 GB (`CONFIG_SFT`): micro-batch 2 con acumulación 64, gradient checkpointing y AdamW con los estados en la RAM (F3). Ninguna cambia el método.

Escribe en `checkpoints/latentr3/<categoría>/sft/`:
- `trainer/`: checkpoints por época (~9 GB cada uno, como máximo 2) para retomar; se borran al terminar.
- `modelo/`: la mejor época (~3,1 GB), con LatentRATT y el tokenizer con `<|Thought|>`. Lo usan la evaluación y el RL.
- `entrenamiento.json`: loss de `valid` por época, tiempos y memoria. Marca el SFT como terminado.

In [14]:
registros_sft = {}
for nombre in DATASETS:
    categoria = categorias[nombre]
    registros_sft[nombre] = lr3.entrenar_sft(categoria, base, lr3.rutas(os.path.join(SALIDA, categoria))['sft'], **CONFIG_SFT)
pd.DataFrame({nombre: {'eval_loss por época': [round(e['eval_loss'], 4) for e in r['evaluaciones']],
                       'mejor eval_loss': r['mejor_eval_loss'], 'pasos': r['pasos_totales'],
                       'h (última sesión)': r['segundos_esta_sesion'] / 3600, 'memoria máx. GB': r['memoria_max_gb']}
              for nombre, r in registros_sft.items()}).T

100%|██████████| 6737/6737 [00:02<00:00, 2547.65it/s]
Some weights of LatentModel were not initialized from the model checkpoint at Qwen/Qwen2.5-1.5B and are newly initialized: ['attention.key.bias', 'attention.key.weight', 'attention.query.bias', 'attention.query.weight', 'attention.value.bias', 'attention.value.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.440600,0.731066
2,0.313100,0.620501


[W1125 01:44:02.191587344 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 759169024 bytes (free: 167772160, total: 17095983104).
There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


,eval_loss por época,mejor eval_loss,pasos,h (última sesión),memoria máx. GB
toys_data,"[0.7311, 0.5729, 0.6205]",0.572852,1263,6.729186,9.759859


### Evaluación del SFT (fila "w/o RL (only SFT)" de la Tabla 2)

Beam search de 10 beams restringido al catálogo con el trie, `length_penalty` 1,0 y lotes de 12 (`BATCH_GEN_SFT`). `latent_train.sh` usa 24, pero con 24 no cabe en 16 GB; 12 es el lote que usan los autores para evaluar el RL (desviación, ver README). Se guarda por partes de 256 filas en `sft/predicciones/test/`. Según el autor (issue #2), esta es la referencia que hay que alcanzar antes de pasar al RL.

In [4]:
for nombre in DATASETS:
    r = lr3.rutas(os.path.join(SALIDA, categorias[nombre]))
    lr3.evaluar_categoria(categorias[nombre], base, r['modelo_sft'], r['pred_sft'], lr3.BATCH_GEN_SFT,
                          lr3.LENGTH_PENALTY['latentr3'])

100%|██████████| 6738/6738 [00:02<00:00, 3287.32it/s]
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
`generation_config` default values have been modified to match model-specific defaults: {'bos_token_id': 151643}. If this is not desired, please set these values explicitly.


Toys_and_Games: parte 1/27, 0.353 s por fila
Toys_and_Games: parte 2/27, 0.344 s por fila
Toys_and_Games: parte 3/27, 0.325 s por fila
Toys_and_Games: parte 4/27, 0.360 s por fila
Toys_and_Games: parte 5/27, 0.324 s por fila
Toys_and_Games: parte 6/27, 0.339 s por fila
Toys_and_Games: parte 7/27, 0.335 s por fila
Toys_and_Games: parte 8/27, 0.350 s por fila
Toys_and_Games: parte 9/27, 0.356 s por fila
Toys_and_Games: parte 10/27, 0.337 s por fila


[W1125 09:26:39.062049109 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 635437056 bytes (free: 614465536, total: 17095983104).


Toys_and_Games: parte 11/27, 0.363 s por fila
Toys_and_Games: parte 12/27, 0.351 s por fila
Toys_and_Games: parte 13/27, 0.355 s por fila
Toys_and_Games: parte 14/27, 0.330 s por fila
Toys_and_Games: parte 15/27, 0.334 s por fila
Toys_and_Games: parte 16/27, 0.335 s por fila
Toys_and_Games: parte 17/27, 0.328 s por fila
Toys_and_Games: parte 18/27, 0.340 s por fila
Toys_and_Games: parte 19/27, 0.364 s por fila
Toys_and_Games: parte 20/27, 0.340 s por fila
Toys_and_Games: parte 21/27, 0.344 s por fila
Toys_and_Games: parte 22/27, 0.356 s por fila
Toys_and_Games: parte 23/27, 0.375 s por fila
Toys_and_Games: parte 24/27, 0.339 s por fila
Toys_and_Games: parte 25/27, 0.363 s por fila
Toys_and_Games: parte 26/27, 0.367 s por fila
Toys_and_Games: parte 27/27, 0.344 s por fila


## Etapa 2: LR-GRPO

Desde el modelo del SFT, solo se entrena LatentRATT (el LLM, los embeddings y `lm_head` quedan congelados). Por cada prompt se toman K = 8 muestras del token latente: la primera sin ruido y las demás con ruido gaussiano de σ = 1,5. La recompensa es −PPL del título objetivo con cada muestra (Eq. 6), sin generar. La ventaja es la recompensa menos la media de las muestras sin ruido del batch de grupos, dividida por su norma (Eq. 7). Una época, 256 filas por paso (32 prompts × 8), lr 1e-5 (de la grilla del paper; 5e-4 y 1e-4 divergieron en Toys, ver README), coseno con 200 pasos de warmup, β = 0 (`attention_grpo.sh`, `train_noise_grpo.py`).

**2 GPUs simuladas (P3):** cada micro-paso trae los mismos 2 grupos que recibían las 2 GPUs de los autores. Cada grupo se colaciona y puntúa por separado con el código original, la ventaja se calcula sobre los 2 grupos y se hace el backward de cada grupo. El gradiente es el mismo que con las 2 GPUs (V7, V8).

Escribe en `checkpoints/latentr3/<categoría>/rl_1e-5/`: `trainer/` (checkpoint cada 200 pasos para retomar; se borra al terminar), `modelo/` (el modelo tras 1 época, como los autores) y `entrenamiento.json` con la recompensa por paso.

In [4]:
registros_rl = {}
for nombre in DATASETS:
    categoria = categorias[nombre]
    r = lr3.rutas(os.path.join(SALIDA, categoria))
    registros_rl[nombre] = lr3.entrenar_rl(categoria, base, r['rl'], r['modelo_sft'], **CONFIG_RL)
pd.DataFrame({nombre: {'pasos': r['pasos_totales'], 'h (última sesión)': r['segundos_esta_sesion'] / 3600,
                       's por paso': r['s_por_paso_mediana'], 'memoria máx. GB': r['memoria_max_gb'],
                       'recompensa (primeros 50 pasos)': sum(r['recompensa_por_paso'][:50]) / max(1, len(r['recompensa_por_paso'][:50])),
                       'recompensa (últimos 50 pasos)': sum(r['recompensa_por_paso'][-50:]) / max(1, len(r['recompensa_por_paso'][-50:]))}
              for nombre, r in registros_rl.items()}).T

100%|██████████| 53898/53898 [00:18<00:00, 2956.69it/s]
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Step,Training Loss
1,0.000600
2,-0.001100
3,-0.003000
4,-0.001100
5,-0.000900
6,-0.006600
7,0.001700
8,0.010600
9,0.001500
10,-0.007000


[W1125 15:04:01.632892751 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 46137344 bytes (free: 0, total: 17095983104).


,pasos,h (última sesión),s por paso,memoria máx. GB,recompensa (primeros 50 pasos),recompensa (últimos 50 pasos)
toys_data,1684.0,6.769579,14.428643,9.620898,-1.608303,-1.558906


### Evaluación del RL y de D3 + LatentR3

La misma decodificación restringida, con lotes de 12 como `attention_grpo.sh`. Se evalúa dos veces el mismo modelo:
- `length_penalty` 1,0: LatentR3 sobre BIGRec, en `rl_1e-5/predicciones/test/`;
- `length_penalty` 0,0: D3 + LatentR3 sin el ensemble con un modelo tradicional, en `rl_1e-5/predicciones/test_d3/`. Que los autores la corrieran así es una deducción del proyecto (base: `evaluate2.sh` de D3, sin `guidance_scale`).

In [5]:
for nombre in DATASETS:
    r = lr3.rutas(os.path.join(SALIDA, categorias[nombre]))
    lr3.evaluar_categoria(categorias[nombre], base, r['modelo_rl'], r['pred_rl'], lr3.BATCH_GEN_RL, lr3.LENGTH_PENALTY['latentr3'])
    lr3.evaluar_categoria(categorias[nombre], base, r['modelo_rl'], r['pred_d3'], lr3.BATCH_GEN_RL, lr3.LENGTH_PENALTY['d3'])

100%|██████████| 6738/6738 [00:02<00:00, 2909.83it/s]
`generation_config` default values have been modified to match model-specific defaults: {'bos_token_id': 151643}. If this is not desired, please set these values explicitly.


Toys_and_Games: parte 1/27, 0.351 s por fila
Toys_and_Games: parte 2/27, 0.344 s por fila
Toys_and_Games: parte 3/27, 0.327 s por fila
Toys_and_Games: parte 4/27, 0.361 s por fila


[W1125 21:57:00.228695273 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 627048448 bytes (free: 27262976, total: 17095983104).


Toys_and_Games: parte 5/27, 0.444 s por fila
Toys_and_Games: parte 6/27, 0.340 s por fila
Toys_and_Games: parte 7/27, 0.334 s por fila
Toys_and_Games: parte 8/27, 0.350 s por fila
Toys_and_Games: parte 9/27, 0.383 s por fila
Toys_and_Games: parte 10/27, 0.337 s por fila


[W1125 22:05:51.072746984 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 635437056 bytes (free: 0, total: 17095983104).


Toys_and_Games: parte 11/27, 0.457 s por fila
Toys_and_Games: parte 12/27, 0.352 s por fila
Toys_and_Games: parte 13/27, 0.360 s por fila
Toys_and_Games: parte 14/27, 0.332 s por fila
Toys_and_Games: parte 15/27, 0.336 s por fila
Toys_and_Games: parte 16/27, 0.335 s por fila
Toys_and_Games: parte 17/27, 0.326 s por fila
Toys_and_Games: parte 18/27, 0.340 s por fila
Toys_and_Games: parte 19/27, 0.469 s por fila
Toys_and_Games: parte 20/27, 0.349 s por fila
Toys_and_Games: parte 21/27, 0.589 s por fila
Toys_and_Games: parte 22/27, 0.398 s por fila
Toys_and_Games: parte 23/27, 0.378 s por fila
Toys_and_Games: parte 24/27, 0.340 s por fila
Toys_and_Games: parte 25/27, 0.366 s por fila
Toys_and_Games: parte 26/27, 0.367 s por fila
Toys_and_Games: parte 27/27, 0.341 s por fila


100%|██████████| 6738/6738 [00:02<00:00, 2983.91it/s]


Toys_and_Games: parte 1/27, 0.337 s por fila
Toys_and_Games: parte 2/27, 0.342 s por fila
Toys_and_Games: parte 3/27, 0.324 s por fila
Toys_and_Games: parte 4/27, 0.359 s por fila
Toys_and_Games: parte 5/27, 0.324 s por fila
Toys_and_Games: parte 6/27, 0.336 s por fila
Toys_and_Games: parte 7/27, 0.329 s por fila
Toys_and_Games: parte 8/27, 0.347 s por fila
Toys_and_Games: parte 9/27, 0.354 s por fila
Toys_and_Games: parte 10/27, 0.335 s por fila


[W1125 22:46:52.636713547 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 635437056 bytes (free: 394264576, total: 17095983104).


Toys_and_Games: parte 11/27, 0.360 s por fila
Toys_and_Games: parte 12/27, 0.351 s por fila
Toys_and_Games: parte 13/27, 0.357 s por fila
Toys_and_Games: parte 14/27, 0.331 s por fila
Toys_and_Games: parte 15/27, 0.337 s por fila
Toys_and_Games: parte 16/27, 0.336 s por fila
Toys_and_Games: parte 17/27, 0.327 s por fila
Toys_and_Games: parte 18/27, 0.338 s por fila
Toys_and_Games: parte 19/27, 0.363 s por fila
Toys_and_Games: parte 20/27, 0.340 s por fila
Toys_and_Games: parte 21/27, 0.346 s por fila
Toys_and_Games: parte 22/27, 0.357 s por fila
Toys_and_Games: parte 23/27, 0.376 s por fila
Toys_and_Games: parte 24/27, 0.338 s por fila
Toys_and_Games: parte 25/27, 0.366 s por fila
Toys_and_Games: parte 26/27, 0.366 s por fila
Toys_and_Games: parte 27/27, 0.338 s por fila


## Métricas

HR@K y NDCG@K con el criterio de `calc.py`: acierto si el título generado, normalizado, es igual al del objetivo. Cada `final_result.json` se verifica corriendo `gao` de `calc.py` sobre el mismo archivo; los títulos fuera del catálogo deben ser 0. La tabla usa todos los datasets con predicciones, no solo los de `DATASETS`.

In [6]:
VARIANTES = {'only SFT': 'pred_sft', 'LatentR3 (RL)': 'pred_rl', 'D3 + LatentR3': 'pred_d3'}
resultados = {}
for nombre, categoria in categorias.items():
    r = lr3.rutas(os.path.join(SALIDA, categoria))
    info = lr3.buscar_archivo(base, 'info', categoria)
    for variante, clave in VARIANTES.items():
        ruta = os.path.join(r[clave], 'final_result.json')
        if os.path.isfile(ruta):
            lr3.verificar_con_calc(ruta, info, calc_py)
            resultados[(nombre, variante)] = lr3.metricas(ruta, info)
assert all(v['fuera del catálogo'] == 0 for v in resultados.values())
pd.DataFrame(resultados).T.round(4)

HR@1    HR@3    HR@5   HR@10   HR@20  NDCG@1  \
toys_data only SFT       0.0359  0.0592  0.0760  0.1067  0.1067  0.0359   
          LatentR3 (RL)  0.0361  0.0592  0.0767  0.1067  0.1067  0.0361   
          D3 + LatentR3  0.0405  0.0727  0.0901  0.1141  0.1141  0.0405   

                         NDCG@3  NDCG@5  NDCG@10  NDCG@20  fuera del catálogo  \
toys_data only SFT       0.0493  0.0561   0.0660   0.0660                 0.0   
          LatentR3 (RL)  0.0494  0.0565   0.0662   0.0662                 0.0   
          D3 + LatentR3  0.0591  0.0662   0.0740   0.0740                 0.0   

                         repetidos   filas  
toys_data only SFT             0.0  6738.0  
          LatentR3 (RL)        0.0  6738.0  
          D3 + LatentR3        0.0  6738.0

## Comparación con el paper

- **Tabla 1:** BIGRec + LatentR3 y D3 + LatentR3 (una corrida determinista, Apéndice E).
- **Tabla 2:** "w/o RL (only SFT)", solo Toys y CDs.
- **Issue #2, CDs:** el SFT de LatentR3 reproducido por un usuario con el código de los autores en 2 GPUs.
- **Nuestro BIGRec y D3** del mismo dataset, si ya están evaluados (`checkpoints/bigrec/`). La mejora relativa de LatentR3 se mide contra ellos: nuestro BIGRec de Toys ya está cerca del "only SFT" del paper, así que la comparación justa es con el mismo pipeline.

In [7]:
METRICAS = ['HR@5', 'HR@10', 'NDCG@5', 'NDCG@10']
PAPER = {('toys_data', 'Tabla 1: BIGRec + LatentR3'): [0.0821, 0.1107, 0.0600, 0.0693],
         ('toys_data', 'Tabla 1: D3 + LatentR3'): [0.0898, 0.1152, 0.0670, 0.0752],
         ('toys_data', 'Tabla 2: w/o RL (only SFT)'): [0.0804, 0.1067, 0.0584, 0.0669],
         ('cd_data', 'Tabla 1: BIGRec + LatentR3'): [0.0934, 0.1160, 0.0754, 0.0826],
         ('cd_data', 'Tabla 1: D3 + LatentR3'): [0.1137, 0.1327, 0.0915, 0.0977],
         ('cd_data', 'Tabla 2: w/o RL (only SFT)'): [0.0830, 0.1005, 0.0662, 0.0719],
         ('cd_data', 'Issue #2: only SFT reproducido'): [0.07698554, 0.09079097, 0.0639346, 0.06841407],
         ('games_data', 'Tabla 1: BIGRec + LatentR3'): [0.0580, 0.0870, 0.0413, 0.0506],
         ('games_data', 'Tabla 1: D3 + LatentR3'): [0.0716, 0.1006, 0.0507, 0.0601],
         ('music_data', 'Tabla 1: BIGRec + LatentR3'): [0.1029, 0.1214, 0.0882, 0.0941],
         ('music_data', 'Tabla 1: D3 + LatentR3'): [0.1066, 0.1229, 0.0920, 0.0973]}
filas = []
for nombre in sorted({n for n, _ in resultados}):
    categoria = categorias[nombre]
    for (n, fuente), valores in PAPER.items():
        if n == nombre:
            filas.append({'dataset': nombre, 'fuente': fuente, **dict(zip(METRICAS, valores))})
    propios = {}
    for variante, carpeta in [('Nuestro BIGRec', 'bigrec'), ('Nuestro D3', 'd3')]:
        ruta = os.path.join(SALIDA_BIGREC, categoria, 'predicciones', carpeta, 'final_result.json')
        if os.path.isfile(ruta):
            propios[variante] = lr3.metricas(ruta, lr3.buscar_archivo(base, 'info', categoria))
    for variante in VARIANTES:
        if (nombre, variante) in resultados:
            propios[f'Nuestro {variante}'] = resultados[(nombre, variante)]
    for fuente, valores in propios.items():
        fila = {'dataset': nombre, 'fuente': fuente, **{m: valores[m] for m in METRICAS}}
        referencia = propios.get('Nuestro D3' if 'D3 +' in fuente else 'Nuestro BIGRec')
        if referencia is not None and fuente not in ('Nuestro BIGRec', 'Nuestro D3'):
            fila['mejora relativa media'] = sum(valores[m] / referencia[m] - 1 for m in METRICAS) / len(METRICAS)
        filas.append(fila)
pd.DataFrame(filas).set_index(['dataset', 'fuente']).round(4)

HR@5   HR@10  NDCG@5  NDCG@10  \
dataset   fuente                                                        
toys_data Tabla 1: BIGRec + LatentR3  0.0821  0.1107  0.0600   0.0693   
          Tabla 1: D3 + LatentR3      0.0898  0.1152  0.0670   0.0752   
          Tabla 2: w/o RL (only SFT)  0.0804  0.1067  0.0584   0.0669   
          Nuestro BIGRec              0.0770  0.1060  0.0563   0.0657   
          Nuestro D3                  0.0873  0.1153  0.0649   0.0740   
          Nuestro only SFT            0.0760  0.1067  0.0561   0.0660   
          Nuestro LatentR3 (RL)       0.0767  0.1067  0.0565   0.0662   
          Nuestro D3 + LatentR3       0.0901  0.1141  0.0662   0.0740   

                                      mejora relativa media  
dataset   fuente                                             
toys_data Tabla 1: BIGRec + LatentR3                    NaN  
          Tabla 1: D3 + LatentR3                        NaN  
          Tabla 2: w/o RL (only SFT)                    NaN  
          Nuestro BIGRec                                NaN  
          Nuestro D3                                    NaN  
          Nuestro only SFT                          -0.0011  
          Nuestro LatentR3 (RL)                      0.0038  
          Nuestro D3 + LatentR3                      0.0106